# Sleep Health and Lifestyle Analysis

**Exploring the relationship between lifestyle factors and sleep quality, duration, and sleep disorders.**

## Project Overview

This project analyzes a sleep health dataset to understand how occupation, physical activity, stress, BMI, and other lifestyle factors impact sleep quality and the prevalence of sleep disorders like insomnia and sleep apnea.

## Learning Objectives

1. Analyze health and lifestyle data with mixed feature types
2. Explore relationships between multiple lifestyle factors and sleep outcomes
3. Identify risk factors for sleep disorders
4. Work with blood pressure data (string parsing)
5. Create actionable health insights from data

## Business / Research Problem

Sleep disorders affect millions worldwide, impacting productivity, health, and quality of life. Understanding modifiable risk factors helps:
- **Healthcare providers** screen for sleep disorders
- **Employers** design wellness programs
- **Individuals** improve their sleep habits

**Key question:** *Which lifestyle factors most strongly predict poor sleep quality and sleep disorders?*

## Why This Analysis Matters

Poor sleep is linked to obesity, heart disease, depression, and reduced cognitive function. Identifying and addressing modifiable risk factors can have significant public health impact.

## Dataset Overview

| Feature | Description |
|---------|-------------|
| `Person ID` | Unique identifier |
| `Gender` | Male/Female |
| `Age` | Age in years |
| `Occupation` | Job type |
| `Sleep Duration` | Hours of sleep per day |
| `Quality of Sleep` | Subjective quality (1-10) |
| `Physical Activity Level` | Minutes of activity per day |
| `Stress Level` | Subjective stress (1-10) |
| `BMI Category` | Weight category |
| `Blood Pressure` | Systolic/Diastolic |
| `Heart Rate` | Resting heart rate (bpm) |
| `Daily Steps` | Steps per day |
| `Sleep Disorder` | None/Insomnia/Sleep Apnea |

## Dataset Source and License Notes- **Source:** [Kaggle - Sleep Health and Lifestyle](https://www.kaggle.com/datasets/uom190346a/sleep-health-and-lifestyle-dataset)- **License:** CC0 Public Domain- **Note:** Synthetic dataset for educational purposes

## Environment Setup

In [1]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

All imports loaded successfully.


## Configuration / Constants

In [3]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'uom190346a/sleep-health-and-lifestyle-dataset'
SIGNIFICANCE_LEVEL = 0.05
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [4]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head()

  0%|          | 0.00/2.54k [00:00<?, ?B/s]

100%|██████████| 2.54k/2.54k [00:00<00:00, 1.90MB/s]

Extracting files...


Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\uom190346a\sleep-health-and-lifestyle-dataset\versions\2
Loaded 374 rows and 13 columns


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


## Data Validation Checks

In [5]:
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing:{df.isnull().sum()[df.isnull().sum() > 0]}')
if df.isnull().sum().sum() == 0: print('No missing values.')
print(f'\nDuplicates: {df.duplicated().sum()}')
df.describe()

Shape: (374, 13)

Data Types:
Person ID                    int64
Gender                      object
Age                          int64
Occupation                  object
Sleep Duration             float64
Quality of Sleep             int64
Physical Activity Level      int64
Stress Level                 int64
BMI Category                object
Blood Pressure              object
Heart Rate                   int64
Daily Steps                  int64
Sleep Disorder              object
dtype: object

Missing:Sleep Disorder    219
dtype: int64

Duplicates: 0


,Person ID,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps
count,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000
mean,187.500000,42.184492,7.132086,7.312834,59.171123,5.385027,70.165775,6816.844920
std,108.108742,8.673133,0.795657,1.196956,20.830804,1.774526,4.135676,1617.915679
min,1.000000,27.000000,5.800000,4.000000,30.000000,3.000000,65.000000,3000.000000
25%,94.250000,35.250000,6.400000,6.000000,45.000000,4.000000,68.000000,5600.000000
50%,187.500000,43.000000,7.200000,7.000000,60.000000,5.000000,70.000000,7000.000000
75%,280.750000,50.000000,7.800000,8.000000,75.000000,7.000000,72.000000,8000.000000
max,374.000000,59.000000,8.500000,9.000000,90.000000,8.000000,86.000000,10000.000000


## Data Cleaning

In [6]:
df = df.drop_duplicates()

# Parse Blood Pressure into systolic and diastolic
bp_col = [c for c in df.columns if 'blood' in c.lower() or 'pressure' in c.lower()]
if bp_col:
    bp = bp_col[0]
    df[['BP_Systolic', 'BP_Diastolic']] = df[bp].str.split('/', expand=True).astype(float)
    print(f'Parsed blood pressure into BP_Systolic and BP_Diastolic')

# Fill Sleep Disorder NaN with 'None'
disorder_col = [c for c in df.columns if 'disorder' in c.lower()]
if disorder_col:
    df[disorder_col[0]] = df[disorder_col[0]].fillna('None')
    print(f'Sleep disorders: {df[disorder_col[0]].value_counts().to_dict()}')

print(f'Final rows: {len(df)}')

Parsed blood pressure into BP_Systolic and BP_Diastolic
Sleep disorders: {'None': 219, 'Sleep Apnea': 78, 'Insomnia': 77}
Final rows: 374


## Exploratory Data Analysis

In [7]:
sleep_dur_col = [c for c in df.columns if 'duration' in c.lower()][0]
quality_col = [c for c in df.columns if 'quality' in c.lower()][0]
stress_col = [c for c in df.columns if 'stress' in c.lower()][0]
activity_col = [c for c in df.columns if 'activity' in c.lower()][0]

print(f'=== Sleep Duration: mean={df[sleep_dur_col].mean():.1f}h, median={df[sleep_dur_col].median():.1f}h ===')
print(f'=== Sleep Quality: mean={df[quality_col].mean():.1f}, median={df[quality_col].median():.1f} ===')
print(f'=== Stress Level: mean={df[stress_col].mean():.1f} ===')

# By occupation
occ_col = [c for c in df.columns if 'occupation' in c.lower()]
if occ_col:
    print(f'\n=== Average Sleep by Occupation ===')
    print(df.groupby(occ_col[0])[sleep_dur_col].mean().sort_values(ascending=False).round(2))

=== Sleep Duration: mean=7.1h, median=7.2h ===
=== Sleep Quality: mean=7.3, median=7.0 ===
=== Stress Level: mean=5.4 ===

=== Average Sleep by Occupation ===
Occupation
Engineer                7.99
Lawyer                  7.41
Accountant              7.11
Nurse                   7.06
Doctor                  6.97
Manager                 6.90
Software Engineer       6.75
Teacher                 6.69
Salesperson             6.40
Scientist               6.00
Sales Representative    5.90
Name: Sleep Duration, dtype: float64


## Univariate Analysis

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(df[sleep_dur_col], bins=15, color='steelblue', edgecolor='white')
axes[0,0].set_title('Sleep Duration Distribution')

axes[0,1].hist(df[quality_col], bins=10, color='coral', edgecolor='white')
axes[0,1].set_title('Sleep Quality Distribution')

axes[0,2].hist(df[stress_col], bins=10, color='mediumseagreen', edgecolor='white')
axes[0,2].set_title('Stress Level Distribution')

axes[1,0].hist(df[activity_col], bins=15, color='mediumpurple', edgecolor='white')
axes[1,0].set_title('Physical Activity Level')

if 'Heart Rate' in df.columns:
    axes[1,1].hist(df['Heart Rate'], bins=15, color='orange', edgecolor='white')
    axes[1,1].set_title('Heart Rate Distribution')

if disorder_col:
    df[disorder_col[0]].value_counts().plot(kind='bar', ax=axes[1,2], color=sns.color_palette('Set2'))
    axes[1,2].set_title('Sleep Disorder Distribution')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if disorder_col:
    dc = disorder_col[0]
    sns.boxplot(data=df, x=dc, y=sleep_dur_col, ax=axes[0], palette='Set2')
    axes[0].set_title('Sleep Duration by Disorder')
    
    sns.boxplot(data=df, x=dc, y=quality_col, ax=axes[1], palette='Set2')
    axes[1].set_title('Sleep Quality by Disorder')
    
    sns.boxplot(data=df, x=dc, y=stress_col, ax=axes[2], palette='Set2')
    axes[2].set_title('Stress Level by Disorder')

plt.tight_layout()
plt.show()

In [10]:
# Scatter: stress vs quality
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(df[stress_col], df[quality_col], alpha=0.5, s=20, c='coral')
axes[0].set_xlabel('Stress Level')
axes[0].set_ylabel('Sleep Quality')
axes[0].set_title('Stress vs Sleep Quality')

axes[1].scatter(df[activity_col], df[sleep_dur_col], alpha=0.5, s=20, c='steelblue')
axes[1].set_xlabel('Physical Activity (min)')
axes[1].set_ylabel('Sleep Duration (hr)')
axes[1].set_title('Activity vs Sleep Duration')

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [11]:
# BMI category analysis
bmi_col = [c for c in df.columns if 'bmi' in c.lower()]
if bmi_col:
    bc = bmi_col[0]
    print('=== Sleep Quality by BMI Category ===')
    print(df.groupby(bc)[[quality_col, sleep_dur_col, stress_col]].mean().round(2))
    if disorder_col:
        print(f'\n=== Sleep Disorder Prevalence by BMI ===')
        print(pd.crosstab(df[bc], df[disorder_col[0]], normalize='index').round(3))

=== Sleep Quality by BMI Category ===
               Quality of Sleep  Sleep Duration  Stress Level
BMI Category                                                 
Normal                     7.66            7.39          5.13
Normal Weight              7.43            7.33          5.19
Obese                      6.40            6.96          5.70
Overweight                 6.90            6.77          5.73

=== Sleep Disorder Prevalence by BMI ===
Sleep Disorder  Insomnia   None  Sleep Apnea
BMI Category                                
Normal             0.036  0.938        0.026
Normal Weight      0.095  0.810        0.095
Obese              0.400  0.000        0.600
Overweight         0.432  0.128        0.439


## Statistical Checks / Hypothesis Testing

In [12]:
# Does stress significantly affect sleep quality?
r, p = stats.pearsonr(df[stress_col], df[quality_col])
print(f'Stress vs Quality: r={r:.4f}, p={p:.2e}')

r2, p2 = stats.pearsonr(df[activity_col], df[sleep_dur_col])
print(f'Activity vs Duration: r={r2:.4f}, p={p2:.2e}')

# ANOVA: Sleep quality across disorders
if disorder_col:
    dc = disorder_col[0]
    groups = [g[quality_col].values for _, g in df.groupby(dc)]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f'\nANOVA Quality by Disorder: F={f_stat:.2f}, p={p_val:.2e}')

Stress vs Quality: r=-0.8988, p=2.88e-135
Activity vs Duration: r=0.2124, p=3.47e-05

ANOVA Quality by Disorder: F=27.60, p=6.69e-12


In [13]:
num_cols = df.select_dtypes(include=[np.number]).columns
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', center=0,
            fmt='.2f', ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Visual Analysis

In [14]:
if occ_col:
    fig, ax = plt.subplots(figsize=(12, 6))
    occ_order = df.groupby(occ_col[0])[quality_col].mean().sort_values(ascending=False).index
    sns.boxplot(data=df, x=occ_col[0], y=quality_col, order=occ_order, palette='viridis', ax=ax)
    ax.set_title('Sleep Quality by Occupation')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Key Findings

1. **Stress is strongly negatively correlated** with sleep quality
2. **Physical activity positively correlates** with sleep duration and quality
3. **Sleep disorder sufferers** have measurably worse sleep quality and duration
4. **BMI category affects sleep disorder prevalence** — overweight/obese individuals have higher rates
5. **Occupation impacts sleep patterns** — some jobs are associated with worse sleep outcomes
6. **Blood pressure correlates** with sleep disorders, suggesting cardiovascular connections

## Limitations

1. **Synthetic dataset:** Results demonstrate technique but may not reflect real populations
2. **Self-reported measures:** Stress and quality are subjective
3. **No temporal data:** Cannot track sleep changes over time
4. **Limited sample size:** Patterns may not generalize
5. **No dietary data:** Nutrition is a major sleep factor not captured

## Recommendations / Next Steps

1. Build a sleep disorder classifier using these lifestyle features
2. Analyze interaction effects (e.g., does exercise buffer stress impact on sleep?)
3. Create a sleep health risk score

### How to Extend This Analysis
- Collect real sleep tracker data (Fitbit, Apple Watch) for validation
- Build an interactive sleep quality dashboard
- Apply clustering to identify lifestyle archetypes

## Common Mistakes

1. **Treating blood pressure as a single string:** Parse into systolic/diastolic numbers
2. **Ignoring the synthetic nature:** Don't draw clinical conclusions from synthetic data
3. **Confusing correlation with causation:** Exercise correlates with better sleep but causation is complex
4. **Not handling 'None' in Sleep Disorder:** NaN and 'None' have different meanings here
5. **Ignoring occupation as a confounder:** Job stress may drive both exercise habits and sleep quality

## Mini Challenge / Exercises

1. Which occupation has the highest rate of sleep disorders?
2. Is there a significant difference in heart rate between people with and without sleep disorders?
3. Create a composite health score from activity, BMI, and sleep quality
4. What is the optimal daily step count associated with the best sleep quality?
5. Build a logistic regression predicting sleep disorder presence

In [15]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Stress is the enemy of sleep** — it's the strongest negative predictor of quality
- **Physical activity helps** both sleep duration and quality
- **BMI and cardiovascular markers** correlate with sleep disorder risk
- **Occupation matters** — job-related stress translates to sleep impact
- **Modifiable factors exist** — exercise, stress management, and weight control can improve sleep

This analysis highlights actionable lifestyle factors for sleep improvement, demonstrating health data EDA techniques.